In [1]:
import STAGATE_pyG
import glob, os, gc
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import cna
torch.set_default_device('mps')

In [ ]:
def run_stagate(dsetname):
    os.makedirs(f'_embeddings', exist_ok=True)
    d = sc.read_h5ad(f'_data/{dsetname}/100u_spots.h5ad')
    
    split_samples = {}
    for sid in d.obs.sid.unique()[:2]:
        myd = d[d.obs.sid == sid].copy()
        STAGATE_pyG.Cal_Spatial_Net(myd, rad_cutoff=15)
        split_samples[sid] = myd

    allsamples = sc.concat([d for d in split_samples.values()], keys=None)
    allsamples.uns['Spatial_Net'] = pd.concat([
        d.uns['Spatial_Net'] for d in split_samples.values()])
    STAGATE_pyG.Stats_Spatial_Net(allsamples)

    allsamples = STAGATE_pyG.train_STAGATE(allsamples, n_epochs=1000)
    sc.pp.neighbors(allsamples, use_rep='STAGATE')
    sc.tl.umap(allsamples)
    sc.tl.leiden(allsamples, resolution=0.8)

    allsamples.write(f'_embeddings/{dsetname}_stagate_noharm.h5ad')

# Run

In [ ]:
run_stagate('ALZ')

------Calculating spatial graph...
The graph contains 16546 edges, 2176 cells.
7.6039 neighbors per cell on average.
------Calculating spatial graph...


/Users/yakir/miniconda3/envs/stagate/lib/python3.12/site-packages/STAGATE_pyG-1.0.0-py3.12.egg/STAGATE_pyG/utils.py:194: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  plot_df = pd.value_counts(pd.value_counts(adata.uns['Spatial_Net']['Cell1']))
/Users/yakir/miniconda3/envs/stagate/lib/python3.12/site-packages/STAGATE_pyG-1.0.0-py3.12.egg/STAGATE_pyG/utils.py:194: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  plot_df = pd.value_counts(pd.value_counts(adata.uns['Spatial_Net']['Cell1']))


The graph contains 19264 edges, 2564 cells.
7.5133 neighbors per cell on average.
Size of Input:  (4740, 140)


 21%|██        | 208/1000 [00:18<01:05, 12.16it/s]